## 1. Identifikasi variabel-variabel yang akan digunakan sebagai variabel bebas (fitur) dan variabel target (biaya medis personal).

In [1]:
import pandas as pd
import numpy as np

df=pd.read_csv('data/insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. Bagi dataset menjadi data latih (train) dan data uji (test) dengan proporsi yang sesuai.

In [2]:
from sklearn.model_selection import train_test_split

X=df.drop(columns=['charges'])
y=df['charges']

X_train, X_test, y_train, y_test=train_test_split(X,y, train_size=0.8, test_size=0.2, random_state=42)

## 3. Lakukan feature scaling jika diperlukan.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_feature=['age', 'bmi', 'children']
cat_feature=['sex', 'smoker', 'region']

preprocessor=ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_feature),
        ('cat', OneHotEncoder(), cat_feature)
    ]
)

## 4. Buat model multiple linear regression menggunakan Scikit-Learn.



In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
lr_pipeline=Pipeline(
    steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())]
)

## 5&6. Latih model pada data latih dan lakukan prediksi pada data uji. Evaluasi model dengan menghitung metrik seperti R-squared, MSE, dan MAE. Tampilkan hasil evaluasi.

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_pipeline.fit(X_train, y_train)
y_pred=lr_pipeline.predict(X_test)

mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)

rmse=np.sqrt(mse)

r2=r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)

MAE: 4181.194473753655
MSE: 33596915.85136149
RMSE: 5796.284659276275
R-squared: 0.7835929767120722


## 7. Ulagi langkah 4 dengan menggunakan model SVR. Anda dapat bereksperimen dengan dengan melakukan hyperparameter tunning.



In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

svr_pipeline=Pipeline(
    steps=[('preprocessor', preprocessor), ('regressor', SVR())]
)

param_grid={
    "regressor__C": [100, 1000, 2000, 5000],
    "regressor__gamma": ["scale", "auto", 0.01, 0.1],
    "regressor__kernel": ["rbf"],
}

grid_search=GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

In [7]:
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)

best_svr=grid_search.best_estimator_
y_pred_svr=best_svr.predict(X_test)

mae=mean_absolute_error(y_test,y_pred_svr)
mse=mean_squared_error(y_test,y_pred_svr)

rmse=np.sqrt(mse)

r2=r2_score(y_test, y_pred_svr)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
{'regressor__C': 5000, 'regressor__gamma': 0.1, 'regressor__kernel': 'rbf'}
MAE: 1900.6482607742437
MSE: 23516479.314870685
RMSE: 4849.379271089309
R-squared: 0.8485238553068835
